# SSDLite320 증강 실험 A1 (Colab GPU)

기준선 `20261104-ssdlite320-det-v2-baseline-01`과 **증강만 바꿔** 같은 조건으로 다시 학습한다.

| 항목 | 기준선 | **A1** | 근거 |
|---|---|---|---|
| 좌우 뒤집기 | 0 | **0.5** | flip-TTA에서 박스 중심이 1.2% 밀림 → 위치 편향 교정. 박스가 정확히 변환됨 |
| 상하 뒤집기 | 0 | **0.5** | 위와 같음 |
| 밝기 | 0 | **0.15** | test 밝기 중앙값 116 vs train 125 (약 0.93배)를 덮는 범위 |
| 대비 | 0 | **0.2** | test 대비(표준편차) 11.6~32.3이 train 범위와 비슷 → 약하게 |
| early stopping patience | 7 | **10** | 증강을 넣으면 수렴이 느려짐. 기준선도 21에폭까지 오르는 중에 멈췄음 |
| 나머지 (batch 8, lr 0.005, 최대 50에폭, seed 42, 데이터 v2) | 같음 | 같음 | 한 번에 하나만 바꿈 |

흐림·임의 회전·크기/위치 변화는 넣지 않는다. test와 train의 선명도·알약 크기·위치 분포가 거의 같아 메울 차이가 없고, 흐림·회전은 박스 정밀도(AP@0.95)를 해칠 수 있다.

**채택 기준 (학습 전에 정함)**
- best mAP@[0.75:0.95]가 **0.9402 + 0.01 = 0.9502 이상**이면 채택한다. 에폭 사이 흔들림이 ±0.01~0.02라 그보다 작은 차이는 우연과 구분할 수 없다.
- 상위 3개 에폭의 평균, AP@0.95, codeit(대회 제공) val의 AP@0.95도 함께 본다.

**실행 방법**
- VS Code에서 커널 선택 > Colab > **GPU** 서버, 또는 Colab 웹에서 GPU 런타임으로 위에서부터 실행한다.
- 결과는 드라이브 `MyDrive/gitkk_runs/detector/<run_name>/`에 저장된다. 세션이 끊기면 1번 셀부터 다시 실행하면 된다. 이미 복사된 사진은 건너뛰고, `last.pt`가 있으면 이어서 학습한다.
- 각 셀은 문제가 있으면 그 자리에서 멈춘다. 오류가 나면 **처음 멈춘 셀**의 메시지를 본다.

In [ ]:
# 1. GPU 확인과 드라이브 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")  # 로그인 창이 뜨면 드라이브 접근을 허용한다
assert os.path.isdir("/content/drive/MyDrive"), (
    "드라이브 연결 실패. VS Code 명령 팔레트(Cmd+Shift+P)에서 "
    "'Colab: Mount Google Drive to Server...'를 실행한 뒤 이 셀을 다시 실행한다"
)
print("드라이브 연결됨")

In [ ]:
# 2. 레포(main)와 의존성. uv 버전은 pyproject 고정값과 같아야 한다
%cd /content
!test -d gitkk-pill-detection || git clone https://github.com/auranova1125/gitkk-pill-detection.git
%cd /content/gitkk-pill-detection
!git checkout -q main && git pull -q --ff-only && git log --oneline -1
%pip install -q uv==0.13.4
!uv sync --locked -q
!uv run python -c 'import torch; print("cuda =", torch.cuda.is_available())'

In [ ]:
# 3. 실행 설정과 탐지기 v2 위치 찾기
import glob, os
from pathlib import Path

RUN_NAME = "20261104-ssdlite320-det-v2-aug-flipcolor-01"   # 이어서 학습하려면 이름을 바꾸지 않는다
OUTPUT_DIR = Path("/content/drive/MyDrive/gitkk_runs/detector")
DST = Path("/content/detector_v2")                         # 런타임 로컬 디스크 (드라이브보다 훨씬 빠름)
SETTINGS = Path("/content/data_settings_local.json")
A1_YAML = Path("/content/ssd_a1.yaml")

# 기준선 best.pt가 드라이브에 있으면 경로를 적는다. 있으면 8번 셀에서 같은 방식으로 함께 평가한다
BASELINE_CKPT = OUTPUT_DIR / "20261104-ssdlite320-det-v2-baseline-01" / "best.pt"

patterns = [
    "/content/drive/MyDrive/gitkk_archive",
    "/content/drive/MyDrive/*/gitkk_archive",
    "/content/drive/Shareddrives/*/gitkk_archive",
    "/content/drive/.shortcut-targets-by-id/*/gitkk_archive",
    "/content/drive/.shortcut-targets-by-id/*/*/gitkk_archive",
]
candidates = [
    Path(p) / "detector_data" / "1. selected_data" / "v2"
    for pat in patterns for p in glob.glob(pat)
]
candidates = [p for p in candidates if (p / "split_manifest.csv").is_file()]
assert candidates, (
    "탐지기 v2를 찾지 못했다. 드라이브 웹에서 '공유 문서함'의 gitkk_archive를 "
    "우클릭 > 정리 > 바로가기 추가 > '내 드라이브'에 둔 뒤 1번 셀부터 다시 실행한다"
)
SRC = candidates[0]
print("탐지기 v2:", SRC)
print(sorted(os.listdir(SRC)))
print("기준선 best.pt:", BASELINE_CKPT if BASELINE_CKPT.exists() else "드라이브에 없음 → 8번 셀은 기준선 기록값과 비교")

## 4. 사진을 런타임 디스크로 복사

드라이브에서 사진을 한 장씩 읽으면 매우 느리므로 32개를 동시에 복사한다. 10,169장에 **약 7~10분** 걸린다. 이미 복사된 사진은 건너뛴다.
마지막 줄이 `passed ac6618cd 8143 2026`이면 정상이다.

In [ ]:
import json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(SRC / "annotations", DST / "annotations", dirs_exist_ok=True)
for name in ["detector_data_summary.json", "detector_label_map.json", "split_manifest.csv"]:
    shutil.copy(SRC / name, DST / name)

jobs = []
for part in ["train", "val"]:
    (DST / part / "images").mkdir(parents=True, exist_ok=True)
    for name in os.listdir(SRC / part / "images"):
        target = DST / part / "images" / name
        if not target.exists():
            jobs.append((SRC / part / "images" / name, target))

def copy(job):
    shutil.copyfile(*job)

with ThreadPoolExecutor(32) as pool:
    list(tqdm(pool.map(copy, jobs), total=len(jobs), desc="사진 복사"))
for part in ["train", "val"]:
    print(part, len(os.listdir(DST / part / "images")), "장")

SETTINGS.write_text(json.dumps(
    {"package_dir": str(DST), "image_roots": {"select": str(DST), "codeit": str(DST)}},
    ensure_ascii=False, indent=2,
))
check = subprocess.run(
    ["uv", "run", "python", "main.py", "check-detector-data",
     "--settings", str(SETTINGS), "--report", "/content/check_v2.json"],
    capture_output=True, text=True,
)
report = json.load(open("/content/check_v2.json"))
assert report["status"] == "passed", report.get("error") or check.stderr[-2000:]
summary = report["data_summary"]
print(report["status"], summary["split_hash"][:8], summary["num_train_images"], summary["num_val_images"])
assert summary["split_hash"].startswith("ac6618cd"), "기준선과 다른 데이터다 (split_hash 불일치)"

## 5. A1 설정 만들기

레포의 `configs/detector.yaml`을 읽어 **증강·patience·경로만** 바꾼다. 나머지가 기준선과 같은지 셀 안에서 확인한다.
`last.pt`가 드라이브에 있으면 자동으로 이어서 학습한다.

In [ ]:
# 5. 커널이 재시작돼도 이 셀부터 실행 가능 (3번 셀은 먼저 실행)
import yaml
%cd /content/gitkk-pill-detection
assert SETTINGS.is_file(), "사진이 준비돼 있지 않다(런타임 초기화). 4번 셀을 다시 실행한다"

cfg = yaml.safe_load(Path("configs/detector.yaml").read_text())
baseline_same = {"model_name": "ssdlite320_mobilenet_v3_large", "batch_size": 8, "max_epochs": 50,
                 "learning_rate": 0.005, "momentum": 0.9, "weight_decay": 0.0005, "seed": 42,
                 "score_threshold": 0.25, "nms_threshold": 0.45, "pretrained_backbone": True}
diff = {k: (cfg.get(k), v) for k, v in baseline_same.items() if cfg.get(k) != v}
assert not diff, f"레포 기본 설정이 기준선과 다르다: {diff}"

run_dir = OUTPUT_DIR / RUN_NAME
last = run_dir / "last.pt"
if run_dir.exists() and not last.exists():
    raise RuntimeError(f"{run_dir} 가 있는데 last.pt가 없다(첫 에폭 전에 끊김). 폴더를 지우고 다시 실행한다")

cfg.update({
    "run_name": RUN_NAME,
    "output_dir": str(OUTPUT_DIR),
    "data_settings_path": str(SETTINGS),
    "horizontal_flip": 0.5,
    "vertical_flip": 0.5,
    "brightness": 0.15,
    "contrast": 0.2,
    "early_stopping_patience": 10,
    "resume_from": str(last) if last.exists() else None,
    "show_progress": True,
})
A1_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print(A1_YAML.read_text())
print("이어서 학습" if last.exists() else "새로 학습")

## 6. 학습

기준선은 23에폭에서 멈췄다. A1은 patience가 10이라 더 오래 걸릴 수 있다. 셀 끝에 전체 소요 시간이 찍힌다. 기준선은 시간 기록이 없었으므로 보고서용으로 적어 둔다.

In [ ]:
%%time
!cd /content/gitkk-pill-detection && uv run python main.py train-detector --config /content/ssd_a1.yaml

## 7. 기준선과 비교 (학습 기록)

기준선 값은 `20261104-ssdlite320-det-v2-baseline-01/metrics.csv`의 `map_75_95`를 그대로 옮긴 것이다 (1부터 센 에폭 기준).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE_MAP7595 = [0.8116, 0.8536, 0.8793, 0.8318, 0.8818, 0.9091, 0.8997, 0.9147, 0.9075, 0.9123,
                0.8733, 0.9266, 0.9318, 0.9132, 0.9217, 0.9402, 0.9197, 0.9280, 0.9324, 0.9191,
                0.9371, 0.9373, 0.9243]
a1 = pd.read_csv(OUTPUT_DIR / RUN_NAME / "metrics.csv")
a1["ep"] = a1["epoch"] + 1

def top3(values):
    return sum(sorted(values, reverse=True)[:3]) / 3

base_best, a1_best = max(BASE_MAP7595), a1["map_75_95"].max()
a1_best_ep = int(a1.loc[a1["map_75_95"].idxmax(), "ep"])
print(f"기준선  best {base_best:.4f} (16에폭) | 상위3 평균 {top3(BASE_MAP7595):.4f} | {len(BASE_MAP7595)}에폭 학습")
print(f"A1      best {a1_best:.4f} ({a1_best_ep}에폭) | 상위3 평균 {top3(a1['map_75_95'].tolist()):.4f} | {len(a1)}에폭 학습")
print(f"차이    best {a1_best - base_best:+.4f} | 상위3 평균 {top3(a1['map_75_95'].tolist()) - top3(BASE_MAP7595):+.4f}")
print("판정:", "채택 (+0.01 이상)" if a1_best >= base_best + 0.01 else "채택 기준 미달 (+0.01 미만)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(1, len(BASE_MAP7595) + 1), BASE_MAP7595, "o-", label="baseline (no aug)")
ax[0].plot(a1["ep"], a1["map_75_95"], "s-", label="A1 (flip + color)")
ax[0].axhline(base_best + 0.01, color="gray", ls=":", label="adopt line (+0.01)")
ax[0].set_title("validation mAP@[.75:.95]"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(a1["ep"], a1["train_loss"], "s-", color="tab:orange")
ax[1].set_title("A1 train loss"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / RUN_NAME / "compare_baseline.png", dpi=120)
a1[["ep", "train_loss", "map", "map_75", "map_75_95", "mean_average_recall", "recall_after_threshold"]].round(4)

## 8. IoU 기준별 AP와 출처별 AP@0.95

학습 기록에는 AP@0.95와 codeit(대회 제공) 사진만의 점수가 없다. best.pt로 val을 다시 예측해 계산한다.
기준선 보고서와 같은 조건(score 0.25 적용 후 예측, 탐지만 평가)이다.

In [ ]:
import json, subprocess
import pandas as pd

script = r'''
import contextlib, io, json, sys
from pathlib import Path
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
sys.path.insert(0, str(Path.cwd()))  # 레포 루트에서 실행해야 레포 모듈을 찾는다
from configs.detector import load_detector_config
from detector.inference import load_detector_inference_model, predict_detector
from detector.training_data import prepare_detector_data

config_path, package_dir = Path(sys.argv[1]), Path(sys.argv[2])
config = load_detector_config(config_path)
data = prepare_detector_data(config)
gt = json.loads((package_dir / "annotations" / "validation.json").read_text())
for a in gt["annotations"]:
    a["category_id"] = 1
gt["categories"] = [{"id": 1, "name": "pill"}]
gt_ids = {im["id"] for im in gt["images"]}
codeit = {im["id"] for im in gt["images"] if Path(im["file_name"]).name.startswith("codeit_")}
with contextlib.redirect_stdout(io.StringIO()):
    coco = COCO(); coco.dataset = gt; coco.createIndex()

def evaluate(checkpoint):
    model, device = load_detector_inference_model(Path(checkpoint), data.label_map, device=data.device)
    result = predict_detector(model, data.val_loader, device, score_threshold=0.25, detector_name="ssd")
    dets = [{"image_id": p.image_id, "category_id": 1, "bbox": list(p.bbox_xywh), "score": p.detector_score}
            for p in result.predictions]
    assert {d["image_id"] for d in dets} <= gt_ids, "예측 image_id가 정답 파일과 맞지 않는다"
    out = {}
    for name, ids in [("all", gt_ids), ("codeit", codeit), ("aihub", gt_ids - codeit)]:
        with contextlib.redirect_stdout(io.StringIO()):
            e = COCOeval(coco, coco.loadRes(dets), "bbox")
            e.params.iouThrs = np.linspace(0.75, 0.95, 5); e.params.imgIds = sorted(ids)
            e.evaluate(); e.accumulate(); e.summarize()
        per = [float(np.mean(x[x > -1])) for x in e.eval["precision"][:, :, :, 0, 2]]
        out[name] = {"images": len(ids), "map_75_95": e.stats[0], "ap": per}
    return out

results = {name: evaluate(path) for name, path in zip(sys.argv[3::2], sys.argv[4::2])}
print(json.dumps(results))
'''
Path("/content/ap_by_iou.py").write_text(script)

eval_cfg = yaml.safe_load(A1_YAML.read_text())
eval_cfg.update({"resume_from": None, "show_progress": False})
Path("/content/ssd_eval.yaml").write_text(yaml.safe_dump(eval_cfg, allow_unicode=True, sort_keys=False))

args = ["A1", str(OUTPUT_DIR / RUN_NAME / "best.pt")]
if BASELINE_CKPT.exists():
    args += ["baseline", str(BASELINE_CKPT)]
run = subprocess.run(["uv", "run", "python", "/content/ap_by_iou.py", "/content/ssd_eval.yaml", str(DST), *args],
                     capture_output=True, text=True, cwd="/content/gitkk-pill-detection")
assert run.returncode == 0, run.stderr[-3000:]
ap = json.loads(run.stdout.strip().splitlines()[-1])
if "baseline" not in ap:
    # 기준선 보고서 9.3절 값 (로컬 재구성 val 2,027장, 공식 v2보다 1장 많음)
    ap["baseline(보고서)"] = {"all": {"images": 2027, "map_75_95": 0.9400, "ap": [0.9893, 0.9891, 0.9889, 0.9846, 0.7479]},
                             "codeit": {"images": 56, "map_75_95": 0.9288, "ap": [1.0, 1.0, 1.0, 0.9884, 0.6554]},
                             "aihub": {"images": 1971, "map_75_95": 0.9400, "ap": [0.9893, 0.9891, 0.9889, 0.9846, 0.7482]}}
rows = []
for model, by_source in ap.items():
    for source, r in by_source.items():
        rows.append({"model": model, "source": source, "images": r["images"], "mAP@[.75:.95]": r["map_75_95"],
                     **{f"AP@{t:.2f}": v for t, v in zip([0.75, 0.80, 0.85, 0.90, 0.95], r["ap"])}})
table = pd.DataFrame(rows).round(4)
table.to_csv(OUTPUT_DIR / RUN_NAME / "ap_by_iou.csv", index=False)
table

## 9. 결과 해석 체크리스트

- [ ] 7번 판정이 **채택**인가? (best +0.01 이상)
- [ ] 상위 3개 평균도 올랐는가? best 하나만 오르면 우연일 수 있다
- [ ] AP@0.95가 올랐는가? 뒤집기의 목적(위치 편향 교정)이 맞았는지 보는 지표다
- [ ] codeit의 AP@0.95가 올랐는가? test와 같은 출처라 Kaggle과 가장 가깝다 (단, 56장뿐이다)
- [ ] 후반에 val이 떨어지기 시작했는가? 그렇다면 과적합 신호이므로 A2(크기·위치 증강)를 검토할 근거가 된다

## 10. (채택 시) Kaggle 제출

드라이브의 `gitkk_runs/detector/<RUN_NAME>/best.pt`와 `label_map.json`을 로컬 `outputs/detector/<RUN_NAME>/`로 받은 뒤,
`outputs/kaggle_submissions/sub03_det_v2/config.yaml`을 복사해 `detector_*` 경로와 `run_name`만 바꿔 같은 방식(score 0.05)으로 제출한다.
기준선 SSD v2의 Public은 **0.58765**다.